In [1]:
import json
import sys
from pathlib import Path
import importlib
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/phoothwincho/Desktop/Movie Knowledge Assistant


In [3]:
import src.retrieval as retrieval

from src.retrieval import (
    search_movies,
    hybrid_search_movies
)

from src.evaluation import evaluate_retrieval

print("Retrieval module:")
print(retrieval.__file__)

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Retrieval module:
/Users/phoothwincho/Desktop/Movie Knowledge Assistant/src/retrieval.py


In [4]:
from src.retrieval import hybrid_search_movies
from src.reranking import rerank_movies

In [5]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

GROUND_TRUTH_PATH = DATA_DIR / "retrieval_ground_truth.json"

print("Project root:", PROJECT_ROOT)
print("Ground truth:", GROUND_TRUTH_PATH)

Project root: /Users/phoothwincho/Desktop/Movie Knowledge Assistant
Ground truth: /Users/phoothwincho/Desktop/Movie Knowledge Assistant/data/retrieval_ground_truth.json


In [6]:
with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = json.load(f)

print(type(ground_truth))

<class 'list'>


In [7]:
print(json.dumps(ground_truth[:2], indent=2))

[
  {
    "question": "A science fiction movie about space exploration",
    "relevant_movies": [
      "Interstellar",
      "The Martian"
    ]
  },
  {
    "question": "A mind-bending science fiction movie",
    "relevant_movies": [
      "Inception",
      "Tenet"
    ]
  }
]


In [8]:
print("Number of queries:", len(ground_truth))

Number of queries: 30


In [9]:
for i, item in enumerate(ground_truth[:5], start=1):
    print(f"\nQuery {i}:")
    print(item)


Query 1:
{'question': 'A science fiction movie about space exploration', 'relevant_movies': ['Interstellar', 'The Martian']}

Query 2:
{'question': 'A mind-bending science fiction movie', 'relevant_movies': ['Inception', 'Tenet']}

Query 3:
{'question': 'A movie about time travel', 'relevant_movies': ['Back to the Future', 'Looper']}

Query 4:
{'question': 'A superhero movie with Marvel characters', 'relevant_movies': ['Avengers: Endgame', 'Iron Man']}

Query 5:
{'question': 'A superhero movie from DC', 'relevant_movies': ['The Dark Knight', 'Man of Steel']}


In [17]:
def get_query(item):
    return item["question"]


def get_relevant_movies(item):
    return item["relevant_movies"]

In [18]:
def normalize_title(title):
    if title is None:
        return ""

    return str(title).strip().lower()

In [19]:
def hit_rate_at_k(results, relevant_movies, k=5):
    """
    Returns 1 if at least one relevant movie
    appears in the top-k results, otherwise 0.
    """

    retrieved_titles = {
        normalize_title(movie.get("title"))
        for movie in results[:k]
    }

    relevant_titles = {
        normalize_title(title)
        for title in relevant_movies
    }

    return int(
        len(retrieved_titles.intersection(relevant_titles)) > 0
    )

In [20]:
def reciprocal_rank_at_k(results, relevant_movies, k=5):
    """
    Returns reciprocal rank of the first relevant movie
    in the top-k results.

    Example:
        relevant movie at rank 1 -> 1.0
        rank 2 -> 0.5
        rank 3 -> 0.333
        not found -> 0.0
    """

    relevant_titles = {
        normalize_title(title)
        for title in relevant_movies
    }

    for rank, movie in enumerate(results[:k], start=1):

        title = normalize_title(movie.get("title"))

        if title in relevant_titles:
            return 1.0 / rank

    return 0.0

In [21]:
test_results = [
    {"title": "Movie A"},
    {"title": "Interstellar"},
    {"title": "Movie C"},
]

test_relevant = ["Interstellar"]

print(
    "Hit Rate@5:",
    hit_rate_at_k(test_results, test_relevant, k=5)
)

print(
    "MRR@5:",
    reciprocal_rank_at_k(
        test_results,
        test_relevant,
        k=5
    )
)

Hit Rate@5: 1
MRR@5: 0.5


In [22]:
print("Number of ground-truth items:", len(ground_truth))
print("\nFirst item:")
print(ground_truth[0])

print("\nKeys:")
print(ground_truth[0].keys())

Number of ground-truth items: 30

First item:
{'question': 'A science fiction movie about space exploration', 'relevant_movies': ['Interstellar', 'The Martian']}

Keys:
dict_keys(['question', 'relevant_movies'])


In [23]:
test_query = get_query(ground_truth[0])

print("Query:")
print(test_query)

Query:
A science fiction movie about space exploration


In [24]:
hybrid_results = hybrid_search_movies(
    test_query,
    limit=5
)

print("\nHybrid results:")

for rank, movie in enumerate(hybrid_results, start=1):
    print(
        f"{rank}. {movie['title']} "
        f"(RRF={movie['score']:.6f})"
    )


Hybrid results:
1. Interstellar: Nolan's Odyssey (RRF=0.027757)
2. The Midnight Sky (RRF=0.027629)
3. Passengers (RRF=0.027397)
4. Ad Astra (RRF=0.026974)
5. Orbiter 9 (RRF=0.026501)


In [25]:
hybrid_candidates = hybrid_search_movies(
    test_query,
    limit=20
)

In [26]:
reranked_results = rerank_movies(
    test_query,
    hybrid_candidates,
    limit=5
)

In [27]:
print("Hybrid + Reranking results:")

for rank, movie in enumerate(reranked_results, start=1):
    print(
        f"{rank}. {movie['title']} "
        f"(rerank={movie['rerank_score']:.4f})"
    )

Hybrid + Reranking results:
1. The Midnight Sky (rerank=4.8771)
2. Interstellar (rerank=4.6327)
3. Project Gemini (rerank=3.4419)
4. Approaching the Unknown (rerank=3.1394)
5. Ad Astra (rerank=2.9478)


In [28]:
test_relevant = get_relevant_movies(ground_truth[0])

print("Relevant movies:")
print(test_relevant)

Relevant movies:
['Interstellar', 'The Martian']


In [29]:
hybrid_hit = hit_rate_at_k(
    hybrid_results,
    test_relevant,
    k=5
)

hybrid_mrr = reciprocal_rank_at_k(
    hybrid_results,
    test_relevant,
    k=5
)

rerank_hit = hit_rate_at_k(
    reranked_results,
    test_relevant,
    k=5
)

rerank_mrr = reciprocal_rank_at_k(
    reranked_results,
    test_relevant,
    k=5
)

print("Hybrid Hit Rate@5:", hybrid_hit)
print("Hybrid MRR@5:", hybrid_mrr)

print("Hybrid + Reranking Hit Rate@5:", rerank_hit)
print("Hybrid + Reranking MRR@5:", rerank_mrr)

Hybrid Hit Rate@5: 0
Hybrid MRR@5: 0.0
Hybrid + Reranking Hit Rate@5: 1
Hybrid + Reranking MRR@5: 0.5


In [30]:
#run full evaluation
evaluation_results = []

for i, item in enumerate(ground_truth):

    query = get_query(item)
    relevant_movies = get_relevant_movies(item)

    print(f"Evaluating {i + 1}/{len(ground_truth)}: {query}")

    # -------------------------
    # Hybrid Search
    # -------------------------

    hybrid_results = hybrid_search_movies(
        query,
        limit=5
    )

    # -------------------------
    # Hybrid + Reranking
    # -------------------------

    hybrid_candidates = hybrid_search_movies(
        query,
        limit=20
    )

    reranked_results = rerank_movies(
        query,
        hybrid_candidates,
        limit=5
    )

    # -------------------------
    # Metrics
    # -------------------------

    hybrid_hit = hit_rate_at_k(
        hybrid_results,
        relevant_movies,
        k=5
    )

    hybrid_mrr = reciprocal_rank_at_k(
        hybrid_results,
        relevant_movies,
        k=5
    )

    rerank_hit = hit_rate_at_k(
        reranked_results,
        relevant_movies,
        k=5
    )

    rerank_mrr = reciprocal_rank_at_k(
        reranked_results,
        relevant_movies,
        k=5
    )

    evaluation_results.append({
        "query": query,
        "hybrid_hit_rate_at_5": hybrid_hit,
        "hybrid_mrr_at_5": hybrid_mrr,
        "reranked_hit_rate_at_5": rerank_hit,
        "reranked_mrr_at_5": rerank_mrr
    })

Evaluating 1/30: A science fiction movie about space exploration
Evaluating 2/30: A mind-bending science fiction movie
Evaluating 3/30: A movie about time travel
Evaluating 4/30: A superhero movie with Marvel characters
Evaluating 5/30: A superhero movie from DC
Evaluating 6/30: A fantasy movie with magic
Evaluating 7/30: A movie about dinosaurs
Evaluating 8/30: A movie about robots and artificial intelligence
Evaluating 9/30: A movie about survival in space
Evaluating 10/30: A movie about dreams
Evaluating 11/30: A romantic comedy
Evaluating 12/30: A movie about friendship
Evaluating 13/30: A movie based on a true story
Evaluating 14/30: A psychological thriller
Evaluating 15/30: A mystery movie with plot twists
Evaluating 16/30: A crime drama
Evaluating 17/30: A movie directed by Christopher Nolan
Evaluating 18/30: A movie directed by Quentin Tarantino
Evaluating 19/30: A movie starring Leonardo DiCaprio
Evaluating 20/30: A movie starring Tom Hanks
Evaluating 21/30: A movie starring 

In [31]:
#convert result to data frame
results_df = pd.DataFrame(evaluation_results)

results_df.head()

,query,hybrid_hit_rate_at_5,hybrid_mrr_at_5,reranked_hit_rate_at_5,reranked_mrr_at_5
0,A science fiction movie about space exploration,0,0.0,1,0.5
1,A mind-bending science fiction movie,0,0.0,0,0.0
2,A movie about time travel,0,0.0,0,0.0
3,A superhero movie with Marvel characters,0,0.0,0,0.0
4,A superhero movie from DC,0,0.0,0,0.0


In [32]:
#calculate overall matrix
summary = {
    "Hybrid Hit Rate@5": results_df["hybrid_hit_rate_at_5"].mean(),
    "Hybrid MRR@5": results_df["hybrid_mrr_at_5"].mean(),
    "Hybrid + Reranking Hit Rate@5": results_df["reranked_hit_rate_at_5"].mean(),
    "Hybrid + Reranking MRR@5": results_df["reranked_mrr_at_5"].mean(),
}

summary

{'Hybrid Hit Rate@5': 0.2,
 'Hybrid MRR@5': 0.10222222222222223,
 'Hybrid + Reranking Hit Rate@5': 0.16666666666666666,
 'Hybrid + Reranking MRR@5': 0.05944444444444444}

In [33]:
comparison_df = pd.DataFrame([
    {
        "Method": "Hybrid Search",
        "Hit Rate@5": results_df["hybrid_hit_rate_at_5"].mean(),
        "MRR@5": results_df["hybrid_mrr_at_5"].mean()
    },
    {
        "Method": "Hybrid + Reranking",
        "Hit Rate@5": results_df["reranked_hit_rate_at_5"].mean(),
        "MRR@5": results_df["reranked_mrr_at_5"].mean()
    }
])

comparison_df

,Method,Hit Rate@5,MRR@5
0,Hybrid Search,0.200000,0.102222
1,Hybrid + Reranking,0.166667,0.059444


In [34]:
#calculate improvement
hybrid_hit_rate = results_df["hybrid_hit_rate_at_5"].mean()
reranked_hit_rate = results_df["reranked_hit_rate_at_5"].mean()

hybrid_mrr = results_df["hybrid_mrr_at_5"].mean()
reranked_mrr = results_df["reranked_mrr_at_5"].mean()


hit_rate_improvement = (
    (reranked_hit_rate - hybrid_hit_rate)
    / hybrid_hit_rate
    * 100
    if hybrid_hit_rate != 0
    else 0
)

mrr_improvement = (
    (reranked_mrr - hybrid_mrr)
    / hybrid_mrr
    * 100
    if hybrid_mrr != 0
    else 0
)


print(f"Hit Rate@5 improvement: {hit_rate_improvement:.2f}%")
print(f"MRR@5 improvement: {mrr_improvement:.2f}%")

Hit Rate@5 improvement: -16.67%
MRR@5 improvement: -41.85%


In [35]:
results_df["hit_rate_improved"] = (
    results_df["reranked_hit_rate_at_5"]
    > results_df["hybrid_hit_rate_at_5"]
)

results_df["mrr_improved"] = (
    results_df["reranked_mrr_at_5"]
    > results_df["hybrid_mrr_at_5"]
)

In [36]:
print(
    "Queries where Hit Rate improved:",
    results_df["hit_rate_improved"].sum()
)

print(
    "Queries where MRR improved:",
    results_df["mrr_improved"].sum()
)

Queries where Hit Rate improved: 1
Queries where MRR improved: 1


In [37]:
#Find queries where reranking made things worse
worse_results = results_df[
    results_df["reranked_mrr_at_5"]
    < results_df["hybrid_mrr_at_5"]
]

worse_results[
    [
        "query",
        "hybrid_mrr_at_5",
        "reranked_mrr_at_5"
    ]
]

,query,hybrid_mrr_at_5,reranked_mrr_at_5
7,A movie about robots and artificial intelligence,1.000000,0.333333
16,A movie directed by Christopher Nolan,0.333333,0.200000
17,A movie directed by Quentin Tarantino,0.333333,0.250000
19,A movie starring Tom Hanks,0.200000,0.000000
23,A movie about prison escape,1.000000,0.500000
26,A movie about virtual reality,0.200000,0.000000


In [38]:
output_path = DATA_DIR / "reranking_detailed_results.csv"

results_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: /Users/phoothwincho/Desktop/Movie Knowledge Assistant/data/reranking_detailed_results.csv


In [39]:
report_path = DATA_DIR / "reranking_report.csv"

comparison_df.to_csv(
    report_path,
    index=False
)

print("Saved:", report_path)

Saved: /Users/phoothwincho/Desktop/Movie Knowledge Assistant/data/reranking_report.csv


In [40]:
print("=" * 60)
print("RERANKING EVALUATION")
print("=" * 60)

print()

print(comparison_df.to_string(index=False))

print()

print(f"Hit Rate@5 improvement: {hit_rate_improvement:.2f}%")
print(f"MRR@5 improvement: {mrr_improvement:.2f}%")

RERANKING EVALUATION

            Method  Hit Rate@5    MRR@5
     Hybrid Search    0.200000 0.102222
Hybrid + Reranking    0.166667 0.059444

Hit Rate@5 improvement: -16.67%
MRR@5 improvement: -41.85%


In [ ]:
print(json.dumps(ground_truth[:2], indent=2))

[
  {
    "question": "A science fiction movie about space exploration",
    "relevant_movies": [
      "Interstellar",
      "The Martian"
    ]
  },
  {
    "question": "A mind-bending science fiction movie",
    "relevant_movies": [
      "Inception",
      "Tenet"
    ]
  }
]


: 